In [ ]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date, timedelta
import time
import json

# Project paths
PROJECT_ROOT = Path.home() / "wspr-propagation"
SW_DATA_DIR = PROJECT_ROOT / "data" / "spaceweather"
SW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# NOAA Space Weather API base
NOAA_BASE = "https://services.swpc.noaa.gov"

# Date range matching our WSPR fetch schedule
YEAR = 2023

print(f"Space weather data directory: {SW_DATA_DIR}")

In [ ]:
def parse_gfz_index(data: dict, index_name: str) -> pd.DataFrame:
    """Parse a GFZ JSON response into a tidy DataFrame."""
    df = pd.DataFrame({
        "time": pd.to_datetime(data["datetime"]),
        index_name: pd.to_numeric(data[index_name], errors="coerce"),
    })
    if "status" in data:
        df["status"] = data["status"]
    return df.sort_values("time").reset_index(drop=True)

# Re-fetch all four indices
df_kp  = fetch_gfz_index("Kp",   2023)
df_ap  = fetch_gfz_index("ap",   2023)
df_sfi = fetch_gfz_index("Fobs", 2023)
df_sn  = fetch_gfz_index("SN",   2023)

print(f"\nKp range:  {df_kp['Kp'].min():.1f} – {df_kp['Kp'].max():.1f}")
print(f"SFI range: {df_sfi['Fobs'].min():.1f} – {df_sfi['Fobs'].max():.1f}")
print(f"SN range:  {df_sn['SN'].min():.0f} – {df_sn['SN'].max():.0f}")

In [ ]:
def build_spaceweather(df_kp, df_ap, df_sfi, df_sn) -> pd.DataFrame:
    """
    Merge all space weather indices into a single DataFrame.
    Kp/ap are 3-hourly; SFI/SN are daily — forward-fill daily values
    to match 3-hourly cadence.
    """
    # Start with 3-hourly Kp as the backbone
    df = df_kp.copy()
    df = df.merge(df_ap[["time", "ap"]], on="time", how="left")
    
    # Normalize timestamps to UTC for merging
    df["time"] = df["time"].dt.tz_localize(None)
    df["date"] = df["time"].dt.date
    
    df_sfi_daily = df_sfi.copy()
    df_sfi_daily["time"] = df_sfi_daily["time"].dt.tz_localize(None)
    df_sfi_daily["date"] = df_sfi_daily["time"].dt.date

    df_sn_daily = df_sn.copy()
    df_sn_daily["time"] = df_sn_daily["time"].dt.tz_localize(None)
    df_sn_daily["date"] = df_sn_daily["time"].dt.date

    # Merge daily indices by date
    df = df.merge(df_sfi_daily[["date", "Fobs"]], on="date", how="left")
    df = df.merge(df_sn_daily[["date", "SN"]],   on="date", how="left")
    df = df.drop(columns=["date"])
    
    return df.reset_index(drop=True)

df_sw = build_spaceweather(df_kp, df_ap, df_sfi, df_sn)

# Cache to parquet
sw_path = SW_DATA_DIR / "spaceweather_2023.parquet"
df_sw.to_parquet(sw_path, index=False)

print(f"Space weather DataFrame: {len(df_sw):,} rows")
print(f"Columns: {list(df_sw.columns)}")
print(f"Cached to: {sw_path}")
display(df_sw.head(8))

In [ ]:
# Correct NCEI/NGDC URL for historical solar event reports
test_url = (
    "https://www.ngdc.noaa.gov/stp/space-weather/swpc-products/"
    "daily_reports/solar_event_reports/2023/06/20230615events.txt"
)

response = requests.get(test_url, timeout=30)
print(f"Status: {response.status_code}")
if response.status_code == 200:
    print(response.text[:3000])

In [ ]:
import re
from datetime import date

def parse_events_file(text: str, file_date: date) -> pd.DataFrame:
    """
    Parse a NOAA daily solar events file.
    Extracts only XRA (X-ray) events relevant to HF propagation.
    """
    events = []
    
    for line in text.splitlines():
        if line.startswith('#') or line.startswith(':') or not line.strip():
            continue
        if 'XRA' not in line:
            continue
        
        try:
            parts = line.split()
            
            # Remove the '+' flag token if present
            parts = [p for p in parts if p != '+']
            
            # Now layout is fixed:
            # [0]event [1]begin [2]max [3]end [4]obs [5]Q [6]type [7]loc [8]particulars [9]peak_flux
            xra_idx = parts.index('XRA')
            
            def hhmm_to_dt(hhmm: str, d: date):
                hhmm = hhmm.replace('/', '0')
                try:
                    h, m = int(hhmm[:2]), int(hhmm[2:])
                    return pd.Timestamp(d.year, d.month, d.day, h, m)
                except:
                    return pd.NaT
            
            begin_str = parts[1]
            max_str   = parts[2]
            end_str   = parts[3]
            
            particulars   = parts[xra_idx + 2] if xra_idx + 2 < len(parts) else ''
            peak_flux_str = parts[xra_idx + 3] if xra_idx + 3 < len(parts) else ''
            
            flare_class = particulars if re.match(r'[ABCMX]\d', particulars) else None
            
            try:
                peak_flux = float(peak_flux_str)
            except:
                peak_flux = None
            
            events.append({
                "date":        file_date,
                "begin":       hhmm_to_dt(begin_str, file_date),
                "peak":        hhmm_to_dt(max_str,   file_date),
                "end":         hhmm_to_dt(end_str,   file_date),
                "flare_class": flare_class,
                "peak_flux":   peak_flux,
            })
        except Exception:
            continue
    
    return pd.DataFrame(events)

df_flares_test = parse_events_file(response.text, date(2023, 6, 15))
print(f"Flares found: {len(df_flares_test)}")
display(df_flares_test)

In [ ]:
def fetch_flares_year(year: int = 2023) -> pd.DataFrame:
    """
    Fetch and parse all daily solar event files for a year.
    Caches result to parquet.
    """
    cache_path = SW_DATA_DIR / f"flares_{year}.parquet"
    if cache_path.exists():
        print(f"Loading cached flares from {cache_path}")
        return pd.read_parquet(cache_path)
    
    base_url = (
        "https://www.ngdc.noaa.gov/stp/space-weather/swpc-products/"
        "daily_reports/solar_event_reports"
    )
    
    all_flares = []
    d = date(year, 1, 1)
    end = date(year, 12, 31)
    failed = 0
    
    while d <= end:
        url = f"{base_url}/{year}/{d.month:02d}/{d.strftime('%Y%m%d')}events.txt"
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                df = parse_events_file(r.text, d)
                if not df.empty:
                    all_flares.append(df)
            else:
                failed += 1
        except Exception as e:
            failed += 1
        
        d += timedelta(days=1)
        time.sleep(0.5)  # polite delay
    
    if not all_flares:
        print("No flare data retrieved.")
        return pd.DataFrame()
    
    df_all = pd.concat(all_flares, ignore_index=True)
    df_all = df_all.dropna(subset=["flare_class"])
    df_all = df_all.sort_values("peak").reset_index(drop=True)
    
    df_all.to_parquet(cache_path, index=False)
    print(f"\nTotal flares: {len(df_all)}, failed days: {failed}")
    print(f"Classes: {df_all['flare_class'].str[0].value_counts().to_dict()}")
    print(f"Cached to {cache_path}")
    return df_all

print("Starting flare fetch for 2023 — ~3 minutes...")
df_flares = fetch_flares_year(2023)